# HippoRAG with LangGraph

This notebook implements a HippoRAG-style graph retrieval pipeline with local Ollama, personalized PageRank, and answer generation.

In [ ]:
from __future__ import annotations

from typing import TypedDict, List, Dict
import numpy as np
import networkx as nx

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import StateGraph, START, END

# =========================
# 1. STATE
# =========================
class HippoRAGState(TypedDict):
    query: str
    docs: List[Document]
    chunks: List[Document]

    graph: nx.Graph
    node_list: List[str]
    passage_map: Dict[str, List[int]]
    node_embeddings: List[List[float]]

    query_entities: List[str]
    query_nodes: List[str]

    scores: Dict[int, float]
    retrieved_docs: List[Document]

    answer: str

# =========================
# 2. MODELS
# =========================
llm = ChatOllama(model="openvoid/Void-Gemini:latest", temperature=0)
embedder = OllamaEmbeddings(model="qwen3-embedding:0.6b")

In [ ]:
# =========================
# 3. LOAD + CHUNK
# =========================
def load_and_chunk(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(docs)
    return docs, chunks

In [ ]:
# =========================
# 4. BUILD GRAPH (OpenIE style)
# =========================
def build_graph(state: HippoRAGState):
    G = nx.Graph()
    passage_map = {}

    for i, chunk in enumerate(state["chunks"][:30]):
        text = chunk.page_content

        triples = llm.invoke(f"""
        Extract triples in format:
        (subject, relation, object)

        TEXT:
        {text}
        """).content

        lines = triples.split("\n")

        for line in lines:
            if "(" in line and "," in line:
                try:
                    parts = line.strip("()").split(",")
                    s, r, o = [p.strip() for p in parts]

                    G.add_node(s)
                    G.add_node(o)
                    G.add_edge(s, o, relation=r)

                    passage_map.setdefault(s, []).append(i)
                    passage_map.setdefault(o, []).append(i)
                except Exception:
                    continue

    return {
        "graph": G,
        "node_list": list(G.nodes),
        "passage_map": passage_map
    }

# =========================
# 5. ADD SYNONYM EDGES
# =========================
def add_similarity_edges(state: HippoRAGState):
    G = state["graph"]
    nodes = state["node_list"]

    if not nodes:
        return {"graph": G, "node_embeddings": []}

    embeddings = embedder.embed_documents(nodes)

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            sim = np.dot(embeddings[i], embeddings[j]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j])
            )
            if sim > 0.8:
                G.add_edge(nodes[i], nodes[j], relation="similar")

    return {
        "graph": G,
        "node_embeddings": embeddings
    }

In [ ]:
# =========================
# 6. EXTRACT QUERY ENTITIES
# =========================
def extract_query_entities(state: HippoRAGState):
    result = llm.invoke(f"""
    Extract named entities from query:

    {state['query']}
    """).content

    entities = [e.strip() for e in result.split("\n") if e.strip()]
    return {"query_entities": entities}

# =========================
# 7. MAP TO GRAPH NODES
# =========================
def map_to_nodes(state: HippoRAGState):
    nodes = state["node_list"]
    query_nodes = []

    if not nodes:
        return {"query_nodes": query_nodes}

    node_emb = state.get("node_embeddings") or embedder.embed_documents(nodes)

    for ent in state["query_entities"]:
        ent_emb = embedder.embed_query(ent)
        sims = [
            np.dot(ent_emb, node_emb[i]) /
            (np.linalg.norm(ent_emb) * np.linalg.norm(node_emb[i]))
            for i in range(len(nodes))
        ]
        best_idx = int(np.argmax(sims))
        query_nodes.append(nodes[best_idx])

    return {"query_nodes": query_nodes}

# =========================
# 8. PERSONALIZED PAGERANK
# =========================
def run_ppr(state: HippoRAGState):
    G = state["graph"]
    query_nodes = state["query_nodes"]

    if len(G.nodes) == 0 or not query_nodes:
        return {"scores": {}}

    personalization = {node: 0.0 for node in G.nodes}
    for qn in query_nodes:
        if qn in personalization:
            personalization[qn] += 1.0 / len(query_nodes)

    scores = nx.pagerank(G, personalization=personalization, alpha=0.85)
    return {"scores": scores}

In [ ]:
# =========================
# 9. PASSAGE SCORING
# =========================
def rank_passages(state: HippoRAGState):
    scores = state["scores"]
    passage_map = state["passage_map"]

    passage_scores = {}
    for node, score in scores.items():
        if node in passage_map:
            for pid in passage_map[node]:
                passage_scores[pid] = passage_scores.get(pid, 0.0) + score

    top_ids = sorted(passage_scores, key=passage_scores.get, reverse=True)[:5]
    retrieved_docs = [state["chunks"][i] for i in top_ids]
    return {"retrieved_docs": retrieved_docs}

# =========================
# 10. GENERATE ANSWER
# =========================
def generate_answer(state: HippoRAGState):
    context = "\n\n".join(d.page_content for d in state["retrieved_docs"])

    answer = llm.invoke(f"""
    Answer the question using context.

    QUESTION:
    {state['query']}

    CONTEXT:
    {context}
    """).content

    return {"answer": answer}

In [ ]:
# =========================
# 11. INITIALIZE GRAPH + BUILD QUERY WORKFLOW
# =========================
def initialize_hipporag(chunks: List[Document]):
    shared_state = {"chunks": chunks}
    shared_state.update(build_graph(shared_state))
    shared_state.update(add_similarity_edges(shared_state))
    return shared_state

def build_hipporag():
    builder = StateGraph(HippoRAGState)

    builder.add_node("extract_entities", extract_query_entities)
    builder.add_node("map_nodes", map_to_nodes)
    builder.add_node("ppr", run_ppr)
    builder.add_node("rank", rank_passages)
    builder.add_node("generate", generate_answer)

    builder.add_edge(START, "extract_entities")
    builder.add_edge("extract_entities", "map_nodes")
    builder.add_edge("map_nodes", "ppr")
    builder.add_edge("ppr", "rank")
    builder.add_edge("rank", "generate")
    builder.add_edge("generate", END)

    return builder.compile()

app = build_hipporag()
app

In [14]:
# =========================
# 12. MAIN LOOP
# =========================
PDF_PATH = r"F:\resume.pdf"
docs, chunks = load_and_chunk(PDF_PATH)

print("Building HippoRAG graph once...")
shared_graph_state = initialize_hipporag(chunks)

print("\nHippoRAG Ready!\n")

while True:
    query = input("Ask: ").strip()

    if query.lower() == "exit":
        break

    if not query:
        print("Please enter a query.")
        continue

    result = app.invoke({
        "query": query,
        "docs": docs,
        "chunks": chunks,
        "graph": shared_graph_state["graph"],
        "node_list": shared_graph_state["node_list"],
        "passage_map": shared_graph_state["passage_map"],
        "node_embeddings": shared_graph_state["node_embeddings"]
    })

    print("\nANSWER:\n")
    print(result["answer"])
    print("\n" + "=" * 60)


ANSWER:

Based on the context provided, here is everything Sajid Miya knows and his schooling history:

### **Schooling and Education**
*   **BE in Computer Science and Engineering:** Currently attending Thapar Institute of Engineering and Technology, Patiala, Punjab (Expected graduation: 2027). He has a CGPA of **8.9/10**.
*   **St. Xavier’s College, Maitighar, Kathmandu:** Completed studies here in September 2023.
*   **Secondary Education Examination (SEE):** Completed at The Old Capital Secondary School, Raniban-Gorkha in August 2021 with a GPA of **3.95/4.0**.
*   **School Leaving Certificate (SLC):** Achieved a GPA of **3.66/4.0**.

### **Knowledge and Skills**
Sajid has a strong foundation in Computer Engineering with a specific interest in Machine Learning. His technical expertise includes:

**Programming Languages & Databases:**
*   **Major Proficiency:** Python, C++, and MySQL.
*   **Familiarity:** C, HTML, CSS, JavaScript, and PHP.

**Tools and Frameworks:**
*   **Web Devel